# Experiment 2 — the PIRATE/PIRATE+ Jacobian diagnosis

Task document, Section 6.2. PIRATE plugs a trained DnCNN denoiser $\mathsf D$ into an
iterative registration method and treats the update as descent on an objective whose
regularizer has $\mathsf D$ as its proximal operator — a hypothesis its paper never states
or tests. Three nested conditions on $J = \nabla\mathsf D(z)$ decide it at each operating
input $z$:

1. **conservative** — $J$ symmetric ($\rho \approx 0$): $\mathsf D$ is a gradient (Poincaré);
2. **proximal** — additionally $\lambda_{\min}(S) \ge 0$: $\mathsf D = \operatorname{prox}$ of a possibly nonconvex regularizer;
3. **convex-proximal** — additionally $\lambda_{\max}(S) \le 1$: that regularizer is convex.

Everything below is read from `results/probe_metrics.json`, produced by

```
~/miniforge3/envs/lpn_env/bin/python -m pnpreg.probe_run --device mps
```

after its calibration gates passed (three denoisers with known answers plus a
symmetric-surrogate zero-test; see `README.md` and `DESIGN.md`). We probe the
**denoiser** inside PIRATE+, not its deep-equilibrium fixed-point iteration.

In [1]:
import json, os, sys
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
from pnpreg import paths

with open(os.path.join(paths.RESULTS, "probe_metrics.json")) as f:
    M = json.load(f)
print("run of", M["config"]["date"], "| torch", M["config"]["torch"],
      "| device", M["config"]["device"])
print("gates passed:", M["gates_passed"],
      f"({sum(g['ok'] for g in M['gates'])}/{len(M['gates'])} gates)")
print("asymmetry floor:", f"{M['asymmetry_floor']:.2e}")

run of 2026-07-30T16:41:14 | torch 2.8.0 | device mps
gates passed: True (191/191 gates)
asymmetry floor: 1.10e-08


In [2]:
rows = []
for name, rec in M["rows"].items():
    s = rec["summary"]
    rows.append({
        "row": name, "points": s["n_points"],
        "rho_mean": s["rho_mean"], "rho_min": s["rho_min"], "rho_max": s["rho_max"],
        "rho_se_max": s["rho_se_max"],
        "lmin(S)": s["lmin_min"], "lmax(S)": s["lmax_max"],
        "viol2_frac": s["frac_viol2"], "viol2_max": s["max_viol2"],
        "viol3_frac": s["frac_viol3"], "viol3_max": s["max_viol3"],
    })
df = pd.DataFrame(rows).set_index("row")
with pd.option_context("display.float_format", lambda v: f"{v:.4g}"):
    display(df)
print(open(os.path.join(paths.RESULTS, "probe_table.md")).read())

,points,rho_mean,rho_min,rho_max,rho_se_max,lmin(S),lmax(S),viol2_frac,viol2_max,viol3_frac,viol3_max
row,,,,,,,,,,,
cal_mixture,8,0,0,0,0,0.1667,11.28,0,0,0.125,10.28
cal_quadrature,16,7.161e-12,0,3.603e-11,1.68e-12,0.2981,1,0,0,0,0
cal_icnn,8,3.097e-13,2.621e-13,3.561e-13,1.925e-14,0.08163,0.9999,0,0,0,0
cal_icnn32,4,4.53e-07,4.375e-07,4.647e-07,2.131e-08,0.08374,0.9999,0,0,0,0
floor,2,0,0,0,0,-24.7,-5.381,1,24.7,0,0
pirate,8,0.4441,0.444,0.4443,0.0001072,-0.7185,1.144,1,0.7185,1,0.144
pirate_plus,8,0.01555,0.01555,0.01556,2.547e-06,0.9456,1.053,0,0,1,0.05322


| row | pts | rho (mean [min,max] ± SE) | lambda_min(S) | lambda_max(S) | cond2 viol frac(max) | cond3 viol frac(max) |
|---|---|---|---|---|---|---|
| CAL mixture (exact, d=64) | 8 | 0 [0, 0] ± 0 | 0.1667 | 11.2778 | 0.00 (0) | 0.12 (10.3) |
| CAL quadrature u_PM (exact, d=2) | 16 | 7.16e-12 [0, 3.6e-11] ± 2e-12 | 0.2981 | 1.0000 | 0.00 (0) | 0.00 (0) |
| CAL ICNN prox (float64, d=64) | 8 | 3.1e-13 [2.62e-13, 3.56e-13] ± 2e-14 | 0.0816 | 0.9999 | 0.00 (0) | 0.00 (0) |
| CAL ICNN prox (float32, d=64) | 4 | 4.53e-07 [4.37e-07, 4.65e-07] ± 2e-08 | 0.0837 | 0.9999 | 0.00 (0) | 0.00 (0) |
| floor: symmetric surrogate | 2 | 0 [0, 0] ± 0 | -- | -- | -- | -- |
| PIRATE (sigma=1) | 8 | 0.444 [0.444, 0.444] ± 0.0001 | -0.7185 | 1.1440 | 1.00 (0.718) | 1.00 (0.144) |
| PIRATE+ | 8 | 0.0156 [0.0155, 0.0156] ± 3e-06 | 0.9456 | 1.0532 | 0.00 (0) | 1.00 (0.0532) |

asymmetry floor: 1.103e-08



In [3]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# eigenvalue panel: per-point lmin/lmax with Ritz error bars vs the [0,1] band
show = [r for r in ("cal_icnn", "pirate", "pirate_plus") if r in M["rows"]]
colors = {"cal_icnn": "0.6", "pirate": "tab:red", "pirate_plus": "tab:blue"}
for off, name in enumerate(show):
    pts = M["rows"][name]["points"]
    x = np.arange(len(pts)) + off * 0.28
    for key, mk in (("lmin", "v"), ("lmax", "^")):
        vals = [p[key] for p in pts]
        errs = [p["res_" + key] for p in pts]
        ax1.errorbar(x, vals, yerr=errs, fmt=mk, ms=5, capsize=2,
                     color=colors[name], label=f"{name} {key}" if off is not None else None)
ax1.axhspan(0, 1, color="0.92", zorder=0)
ax1.axhline(0, color="0.5", lw=0.8); ax1.axhline(1, color="0.5", lw=0.8)
ax1.set_xlabel("test point"); ax1.set_ylabel("eigenvalue of S")
ax1.set_title("extreme eigenvalues vs the proximal band [0, 1]")
ax1.legend(fontsize=7, ncol=3)

# asymmetry panel: rho per point vs the floor
for name in show:
    pts = M["rows"][name]["points"]
    ax2.errorbar(np.arange(len(pts)), [p["rho"] for p in pts],
                 yerr=[p["rho_se"] for p in pts], fmt="o", ms=5, capsize=2,
                 color=colors[name], label=name)
ax2.axhline(M["asymmetry_floor"], color="k", ls="--", lw=0.8,
            label=f"floor {M['asymmetry_floor']:.1e}")
ax2.set_yscale("log"); ax2.set_xlabel("test point")
ax2.set_ylabel(r"$\rho = \|J-J^T\|_F \, / \, 2\|J\|_F$")
ax2.set_title("asymmetry ratio vs the probe-noise floor")
ax2.legend(fontsize=8)
fig.tight_layout()
os.makedirs(paths.FIGS, exist_ok=True)
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(paths.FIGS, f"experiment2_probe.{ext}"),
                dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1100x400 with 2 Axes>

In [4]:
summary = {name: M["rows"][name]["summary"] for name in M["rows"]}
fl = M["asymmetry_floor"]
for name in ("pirate", "pirate_plus"):
    if name not in summary:
        continue
    s = summary[name]
    sym = "<= floor" if s["rho_mean"] <= fl + 2 * s["rho_se_max"] else f"{s['rho_mean']:.3g} (NONZERO)"
    print(f"{name:12s} rho {sym:24s} lmin {s['lmin_min']:+.4f}  lmax {s['lmax_max']:.4f}  "
          f"viol2 {s['frac_viol2']:.0%} (max {s['max_viol2']:.3g})  "
          f"viol3 {s['frac_viol3']:.0%} (max {s['max_viol3']:.3g})")

pirate       rho 0.444 (NONZERO)          lmin -0.7185  lmax 1.1440  viol2 100% (max 0.718)  viol3 100% (max 0.144)
pirate_plus  rho 0.0156 (NONZERO)         lmin +0.9456  lmax 1.0532  viol2 0% (max 0)  viol3 100% (max 0.0532)


## Reading

By the three conditions: a row with $\rho$ at the floor and $\lambda_{\min} \ge 0$ admits a
(possibly nonconvex) regularizer with $\operatorname{prox} = \mathsf D$, and the Tier-2 recovery
targets exactly that object; $\lambda_{\max} \le 1$ in addition would put it in the convex
class of `tv_pm`. A large $\rho$ means $\mathsf D$ is the gradient of nothing — the objective
PIRATE's paper writes down does not exist for that denoiser, and the honest output is the
theory sections plus this table as a negative result about the hypothesis their analysis
rests on (task document, kill criteria).

The PIRATE row against the PIRATE+ row is the controlled comparison: identical
architecture, weights moved only by the deep-equilibrium fine-tuning. Whatever the
direction of the movement, it is the sharpest content the released weights support —
the released checkpoint set contains a single denoiser (their $\sigma = 1$), so there are
no further $\sigma$ rows without retraining on our side (see `changes.txt` C20).

Full per-point numbers, seeds, budgets, Ritz residuals, and gate records are in
`results/probe_metrics.json`; the paper-ready table is `results/probe_table.tex`.